In [ ]:
# # Don't run, this is necessary to install transformers==3.2.0, but won't run in colab... not sure why
# !sudo apt-get install python3.7
# !sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.7 1
# !sudo update-alternatives --config python3
# !sudo apt install python3-pip
# !python --version
# !sudo apt-get install python3.7-distutils

In [ ]:
# !sudo update-alternatives --config python3

In [ ]:
!pip install datasets==2.10.1 huggingface_hub==0.13.0 
# !pip install git+https://github.com/huggingface/transformers
!pip install transformers==4.27.0
!pip install evaluate

In [2]:
import torch
!pip install --upgrade accelerate

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.3/215.3 kB 5.3 MB/s eta 0:00:00


In [3]:
# Created notebook to save dataset locally and fine-tune model for sentiment analysis
from google.colab import drive
from pathlib import Path
import sys
from datasets import load_dataset

drive.mount('/content/drive/')

Mounted at /content/drive/


In [4]:
strPath = "/content/drive/MyDrive/CMSC848D_Project/Spurious_Correlations-20230425T172048Z-001/Spurious_Correlations/Identifying-and-Mitigating-Spurious-Correlations-for-Improving-Robustness-in-NLP-Models"
# sys.path.insert(0,"drive/MyDrive/Spurious_Correlations/Identifying-and-Mitigating-Spurious-Correlations-for-Improving-Robustness-in-NLP-Models")
filePath = Path(strPath)
%cd $filePath

/content/drive/MyDrive/CMSC848D_Project/Spurious_Correlations-20230425T172048Z-001/Spurious_Correlations/Identifying-and-Mitigating-Spurious-Correlations-for-Improving-Robustness-in-NLP-Models


In [5]:
%load_ext autoreload
%autoreload 2

In [6]:
 %reload_ext autoreload

In [7]:
# %load trainer.py
from utils import trainer
import pandas as pd
import os

In [8]:
# Only use this if you want to create a new dataset, otherwise use the load in sentimement_attention
# Save dataset locally onto drive ... even though data gets cached with load
import os
# dataset = load_dataset("glue", "sst2")
dataset1 = load_dataset("yelp_polarity")
train_df = pd.DataFrame(dataset1['train'])
test_df = pd.DataFrame(dataset1['test'])
# dev_df = pd.DataFrame(dataset['validation'])

if not os.path.exists("transformers/yelp_data/"):
  os.makedirs("transformers/yelp_data/")

if not os.path.exists("models/distilbert_yelp_base_uncased/"):
  os.makedirs("models/distilbert_yelp_base_uncased/")
  
# train_df.to_csv("transformers/glue_data/SST-2/dev.tsv", sep = "\t")
train_df.to_csv("transformers/yelp_data/train.tsv", sep = "\t")
test_df.to_csv("transformers/yelp_data/dev.tsv", sep = "\t")
# dev_df.to_csv("transformers/yelp_data/dev.tsv", sep = "\t")


dataset2 = load_dataset("glue", "sst2")
train_df = pd.DataFrame(dataset2['train'])
test_df = pd.DataFrame(dataset2['test'])
dev_df = pd.DataFrame(dataset2['validation'])

if not os.path.exists("transformers/glue_data/SST-2/"):
  os.makedirs("transformers/glue_data/SST-2/")

if not os.path.exists("models/SST_base_cased/"):
  os.makedirs("models/SST_base_cased/")
  
# train_df.to_csv("transformers/glue_data/SST-2/dev.tsv", sep = "\t")
train_df.to_csv("transformers/glue_data/SST-2/train.tsv", sep = "\t")
test_df.to_csv("transformers/glue_data/SST-2/test.tsv", sep = "\t")
dev_df.to_csv("transformers/glue_data/SST-2/dev.tsv", sep = "\t")

Generating train split:   0%|          | 0/560000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/38000 [00:00<?, ? examples/s]

Dataset yelp_polarity downloaded and prepared to /root/.cache/huggingface/datasets/yelp_polarity/plain_text/1.0.0/14f90415c754f47cf9087eadac25823a395fef4400c7903c5897f55cfaaa6f61. Subsequent calls will reuse this data.


  0%|          | 0/2 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Dataset glue downloaded and prepared to /root/.cache/huggingface/datasets/glue/sst2/1.0.0/dacbe3125aa31d7f70367a07a8a9e72a5a0bfeb5fc42e75c9db75b96da6053ad. Subsequent calls will reuse this data.


  0%|          | 0/3 [00:00<?, ?it/s]

In [9]:
print(dataset1)
print(dataset2)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 560000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 38000
    })
})
DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})


In [ ]:
 %reload_ext autoreload
# if not os.path.exists("models/SST_base_cased/"):
#   os.makedirs("models/SST_base_cased/")
if not os.path.exists("models/distilbert_SST_base_cased/"):
  os.makedirs("models/distilbert_SST_base_cased/")


!python3 utils/run_attention.py --model_name_or_path "distilbert-base-uncased-finetuned-sst-2-english" --task_name "sst-2" --do_eval True --max_seq_length 512  --per_device_train_batch_size 16 --learning_rate 2e-5 --num_train_epochs 3 --data_dir "transformers/glue_data/SST-2" --output_dir "models/distilbert_SST_base_cased" --overwrite_output_dir True --logging_strategy epoch --save_strategy epoch --evaluation_strategy epoch
# !python3 utils/run_attention.py --model_name_or_path "bert-base-cased" --task_name "sst-2" --do_train True --do_eval True --max_seq_length 512  --per_device_train_batch_size 16 --learning_rate 2e-5 --num_train_epochs 3 --data_dir "transformers/glue_data/SST-2/" --output_dir "models/SST_base_cased" --overwrite_output_dir True --logging_strategy epoch --save_strategy epoch --evaluation_strategy epoch

In [10]:
 %reload_ext autoreload
# if not os.path.exists("models/SST_base_cased/"):
#   os.makedirs("models/SST_base_cased/")
if not os.path.exists("models/distilbert_yelp_base_uncased/"):
  os.makedirs("models/distilbert_yelp_base_uncased/")


!python3 utils/run_attention.py --model_name_or_path "randellcotta/distilbert-base-uncased-finetuned-yelp-polarity" --task_name "sst-2" --do_eval True --max_seq_length 512  --per_device_train_batch_size 16 --learning_rate 2e-5 --num_train_epochs 3 --data_dir "transformers/yelp_data" --output_dir "models/distilbert_yelp_base_uncased" --overwrite_output_dir True --logging_strategy epoch --save_strategy epoch --evaluation_strategy epoch
# !python3 utils/run_attention.py --model_name_or_path "bert-base-cased" --task_name "sst-2" --do_train True --do_eval True --max_seq_length 512  --per_device_train_batch_size 16 --learning_rate 2e-5 --num_train_epochs 3 --data_dir "transformers/glue_data/SST-2/" --output_dir "models/SST_base_cased" --overwrite_output_dir True --logging_strategy epoch --save_strategy epoch --evaluation_strategy epoch

2023-04-25 18:46:49.804432: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
04/25/2023 18:46:51 - WARNING - __main__ -   Process rank: -1, device: cuda:0, n_gpu: 1, distributed training: False, 16-bits training: False
04/25/2023 18:46:51 - INFO - __main__ -   Training/evaluation parameters TrainingArguments(
_n_gpu=1,
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_pin_memory=True,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_steps=None,
evaluation_strategy=epoch,
fp16=False,
fp16_backend=auto,
fp16_full_eval=False,
fp16_opt_level=O1,
fsdp=[],
fsdp_config={'fsdp_min_num_params': 0, 'xla': False, 'xla_fsdp_

In [ ]:
import torch
print(train_df['sentence'])
train_X = torch.tensor(train_df['sentence']).values.astype(dtype_string)

In [ ]:
 %reload_ext autoreload
 import torch
!python3 utils/run_glue.py --model_name_or_path "distilbert-base-uncased-finetuned-sst-2-english" --task_name "sst2" --do_eval --max_seq_length 512  --per_device_train_batch_size 16 --learning_rate 2e-5 --num_train_epochs 1  --output_dir "models/distilbert_SST_base_cased" --overwrite_output_dir True --logging_strategy epoch --save_strategy epoch --evaluation_strategy epoch